# Agent 2: Gap Analysis Agent

This notebook implements the complete, detailed implementation of the **Gap Analysis Agent** used in CareerAtlas. The agent retrieves taxonomy skills for a target career path, queries a cached local BM25 corpus and a semantic embedding index, performs Reciprocal Rank Fusion (RRF), cross-encoder reranking, and runs LLM analysis to identify critical gaps.

### Step 1: API Keys Setup
Please configure your API keys here.

In [ ]:
import os
import getpass

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")
if not os.environ.get("JINA_API_KEY"):
    os.environ["JINA_API_KEY"] = getpass.getpass("Enter your Jina API Key (optional): ")

class Settings:
    google_api_key = os.environ.get("GOOGLE_API_KEY", "")
    google_api_keys = [google_api_key] if google_api_key else []
    groq_api_key = os.environ.get("GROQ_API_KEY", "")
    groq_model = "llama-3.3-70b-versatile"
    jina_api_key = os.environ.get("JINA_API_KEY", "")
    pinecone_api_key = os.environ.get("PINECONE_API_KEY", "")
    pinecone_index_name = "careeratlas"
    pinecone_host = ""

settings = Settings()

In [ ]:
# Install required dependencies
# !pip install pydantic rank-bm25 numpy httpx langchain-google-genai langchain-openai pinecone

### Step 2: Imports & Pydantic Schemas
Exact schema definitions from `app.gap_analysis.schemas`.

In [ ]:
import math
import hashlib
import asyncio
from typing import List, Dict, Tuple, Any, Optional, Union
from pydantic import BaseModel, Field, field_validator
from rank_bm25 import BM25Okapi
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_groq import ChatGroq
from pinecone import Pinecone

class GapSchema(BaseModel):
    skill: str = Field(description="Name of the missing skill")
    category: str = Field(description="Category: framework | language | concept | tool | soft")
    relevance: Union[int, str] = Field(description="Relevance percentage to the target role (0-100)")
    difficulty: str = Field(description="One of: Easy, Medium, Hard")
    level_required: str = Field(default="intermediate", description="Required proficiency")
    prerequisites: List[str] = Field(description="List of skills required to learn this")
    why: str = Field(description="1 sentence explaining why this skill is a crucial gap")

    @field_validator("relevance", mode="before")
    @classmethod
    def _coerce_relevance(cls, v):
        if isinstance(v, (int, float)):
            if 0.0 <= v <= 1.0: return int(v * 100)
            return int(v)
        if isinstance(v, str):
            try:
                val = float(v)
                if 0.0 <= val <= 1.0: return int(val * 100)
                return int(val)
            except ValueError:
                digits = "".join(c for c in v if c.isdigit())
                return int(digits) if digits else 0
        return v

class GapAnalysisResponse(BaseModel):
    gaps: List[GapSchema] = Field(description="Up to 6 identified skill gaps")
    justifications: dict[str, str] = Field(description="Mapping skill name to justification")

### Step 3: Embeddings and Reranking Service
Handles Jina Rerank and Google text-embedding API calls.

In [ ]:
def get_pinecone_index():
    if not settings.pinecone_api_key:
        return None
    pc = Pinecone(api_key=settings.pinecone_api_key)
    return pc.Index(settings.pinecone_index_name)

class AIService:
    def __init__(self):
        self.google_api_key = settings.google_api_key
        self.jina_api_key = settings.jina_api_key
        
    async def get_embeddings(self, texts: List[str], task_type: str = "retrieval_query") -> List[List[float]]:
        if not self.google_api_key:
            raise ValueError("GOOGLE_API_KEY not configured")
        prefix = "Represent this query for searching relevant passages: " if task_type == "retrieval_query" else "Represent this document for retrieval: "
        prefixed_texts = [prefix + t for t in texts]
        embeddings = GoogleGenerativeAIEmbeddings(
            model="models/gemini-embedding-2-preview",
            google_api_key=self.google_api_key,
            task_type=task_type
        )
        return await embeddings.aembed_documents(prefixed_texts)

    async def rerank(self, query: str, documents: List[str], top_n: int = 5) -> List[dict]:
        if not self.jina_api_key or not documents:
            return [{"index": i, "relevance_score": 0.9 - 0.1 * i} for i in range(len(documents))]
        import httpx
        headers = {"Content-Type": "application/json", "Authorization": f"Bearer {self.jina_api_key}"}
        payload = {"model": "jina-reranker-v3", "query": query, "documents": documents, "top_n": top_n}
        async with httpx.AsyncClient(timeout=30.0) as client:
            response = await client.post("https://api.jina.ai/v1/rerank", headers=headers, json=payload)
            response.raise_for_status()
            return response.json()["results"]

ai_service = AIService()

### Step 4: On-The-Fly Taxonomy Generation
Implements taxonomy_gen.py. Generates role taxonomies dynamically for uncurated roles.

In [ ]:
class _GenSkill(BaseModel):
    skill_name: str
    description: str
    category: str
    level_required: str
    prerequisites: List[str] = Field(default_factory=list)

class _GenTaxonomy(BaseModel):
    skills: List[_GenSkill]

ROLE_SLUG_MAP = {
    "machine learning engineer": "ml_engineer",
    "data scientist": "data_scientist",
    "frontend engineer": "frontend_engineer",
    "backend engineer": "swe_backend",
    "full-stack engineer": "fullstack_engineer",
    "data analyst": "data_analyst",
    "devops engineer": "devops_engineer",
    "associate product manager": "product_manager",
}

def resolve_role_slug(title: str) -> str:
    normalized = title.strip().lower()
    if normalized in ROLE_SLUG_MAP: return ROLE_SLUG_MAP[normalized]
    return normalized.replace(" ", "_").replace("(", "").replace(")", "")

TAXONOMY_STORE = {}

async def _generate_rows(role_title: str) -> List[_GenSkill]:
    model = ChatGroq(model=settings.groq_model, groq_api_key=settings.groq_api_key, temperature=0.2)
    prompt = (
        f"List the 18-22 most important skills required for a \"{role_title}\". "
        "Categorize each as framework, language, concept, tool, or soft."
    )
    chain = model.with_structured_output(_GenTaxonomy)
    result: _GenTaxonomy = await chain.ainvoke(prompt)
    return result.skills

async def ensure_role_taxonomy(role_title: str) -> str:
    role_slug = resolve_role_slug(role_title)
    if role_slug in TAXONOMY_STORE:
        return "curated"
    
    rows = await _generate_rows(role_title)
    TAXONOMY_STORE[role_slug] = [
        {
            "skill_name": r.skill_name,
            "description": r.description,
            "category": r.category,
            "level_required": r.level_required,
            "prerequisites": r.prerequisites,
            "role": role_slug
        }
        for r in rows
    ]
    
    index = get_pinecone_index()
    if index is not None:
        records = []
        for r in rows:
            text = f"{r.skill_name}: {r.description}"
            vec = (await ai_service.get_embeddings([text], task_type="retrieval_document"))[0]
            h = hashlib.sha1(f"{role_slug}|{r.skill_name}".encode("utf-8")).hexdigest()[:16]
            records.append({
                "id": f"tax_{h}",
                "values": vec,
                "metadata": {
                    "skill_name": r.skill_name, "description": r.description, "category": r.category,
                    "level_required": r.level_required, "prerequisites": r.prerequisites,
                    "role": role_slug, "namespace": "taxonomy", "source": "llm"
                }
            })
        index.upsert(vectors=records, namespace="taxonomy")
    return "generated"

### Step 5: Hybrid Retrieval & RRF
Performs semantic and lexical BM25 matching and reciprocal rank fusion.

In [ ]:
async def hybrid_retrieve(user_skills: List[str], target_role_title: str, user_headline: str = "") -> List[dict]:
    role_slug = resolve_role_slug(target_role_title)
    role_candidates = TAXONOMY_STORE.get(role_slug, [])
    if not role_candidates: return []
    
    # BM25 lexical ranking
    corpus_texts = [f"{c['skill_name']} {c['description']}".lower().split() for c in role_candidates]
    bm25 = BM25Okapi(corpus_texts)
    bm25_query = " ".join(user_skills).lower().split()
    bm25_scores = bm25.get_scores(bm25_query)
    bm25_ranked = [role_candidates[i] for i in sorted(range(len(role_candidates)), key=lambda x: bm25_scores[x], reverse=True)]
    
    # Semantic ranking (mock fallback for local use if Pinecone index is absent)
    index = get_pinecone_index()
    if index is not None:
        query_text = f"Identify essential skills for a {target_role_title}. Headline: {user_headline}. Skills: {', '.join(user_skills)}"
        query_vec = (await ai_service.get_embeddings([query_text], task_type="retrieval_query"))[0]
        res = index.query(vector=query_vec, top_k=20, namespace="taxonomy", filter={"role": {"$eq": role_slug}}, include_metadata=True)
        semantic_ranked = [m["metadata"] for m in res["matches"]]
    else:
        semantic_ranked = role_candidates
    
    k = 60
    scores = {}
    meta_lookup = {s["skill_name"]: s for s in role_candidates}
    
    for rank, s in enumerate(semantic_ranked):
        scores[s["skill_name"]] = scores.get(s["skill_name"], 0) + 1.0 / (k + rank + 1)
    for rank, s in enumerate(bm25_ranked):
        scores[s["skill_name"]] = scores.get(s["skill_name"], 0) + 1.0 / (k + rank + 1)
        
    fused = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    
    # Jina Rerank
    docs = [f"{name}: {meta_lookup[name]['description']}" for name, _ in fused]
    reranked = await ai_service.rerank(f"Skills needed for {target_role_title}", docs, top_n=5)
    
    results = []
    for r in reranked:
        name = fused[r["index"]][0]
        meta = meta_lookup[name].copy()
        meta["relevance_score"] = r["relevance_score"]
        results.append(meta)
        
    return results

### Step 6: Gap Analysis Service
Binds taxonomy check, hybrid retrieval, and Gemini together.

In [ ]:
GAP_ANALYSIS_PROMPT = PromptTemplate.from_template(
    """You are an expert career coach and recruiter.
Identify the most critical SKILL GAPS (up to 6 gaps) for a candidate wanting to become a {target_role}.

## USER CURRENT SKILLS
{user_skills}

## USER HEADLINE
{user_headline}

## RETRIEVED ROLE REQUIREMENTS (ranked by relevance)
{role_requirements}

Rules:
1. Do not list skills they already have.
2. Rank by importance.
"""
)

async def generate_gaps_for_user(user_skills: List[str], target_role_title: str, user_headline: str = "") -> Tuple[List[GapSchema], dict]:
    # 1. ensure role taxonomy exists
    taxonomy_source = await ensure_role_taxonomy(target_role_title)
    
    # 2. run hybrid retrieve
    retrieved = await hybrid_retrieve(user_skills, target_role_title, user_headline)
    
    req_lines = []
    for i, s in enumerate(retrieved, 1):
        raw = s.get("relevance_score", 0)
        norm = 1.0 / (1.0 + math.exp(-5.0 * raw))
        s["relevance_score"] = norm
        req_lines.append(f"{i}. {s['skill_name']} [{s['category']}] (level: {s['level_required']}) - {s['description']} [relevance_score: {norm:.3f}]")
        
    requirements_text = "\n".join(req_lines)
    
    # 3. LLM structured analysis
    model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=settings.google_api_key, temperature=0.0)
    chain = GAP_ANALYSIS_PROMPT | model.with_structured_output(GapAnalysisResponse)
    
    result: GapAnalysisResponse = await chain.ainvoke({
        "target_role": target_role_title,
        "user_skills": ", ".join(user_skills),
        "user_headline": user_headline,
        "role_requirements": requirements_text
    })
    
    return result.gaps, {"retrieved": retrieved, "justifications": result.justifications}

### Step 7: Test Run

In [ ]:
user_skills = ["Python", "Scikit-Learn", "SQL"]
target_role = "Machine Learning Engineer"
user_headline = "Aspiring ML developer"

async def test_run():
    gaps, details = await generate_gaps_for_user(user_skills, target_role, user_headline)
    print(f"Identified Gaps:")
    for g in gaps:
        print(f"- {g.skill} ({g.category}): Relevance={g.relevance}%, Difficulty={g.difficulty}, Why: {g.why}")

try:
    asyncio.run(test_run())
except Exception as e:
    print(f"Execution skipped or failed. Error: {e}")